# 📚 02. 핵심 — pandas와 함께 배우는 Python 심화

이 노트북은 **pandas를 사용하면서 동시에 익혀야 할** 핵심 Python 개념을 다룹니다.

## 학습 목표
| 개념 | pandas 연결 포인트 |
|------|-------------------|
| 함수 (def) | `df.apply(함수)`로 전체 컬럼에 함수 적용 |
| lambda 함수 | `df['열'].map(lambda x: ...)` 인라인 변환 |
| 슬라이싱 | `df.iloc[0:3, 1:4]` 위치 기반 데이터 선택 |
| 인덱싱 | `df.loc['인덱스', '컬럼명']` 이름 기반 선택 |
| 컴프리헨션 | 열 변환, 필터링을 한 줄로 처리 |

> **전제**: 01_필수 노트북을 먼저 완료해주세요.

In [ ]:
# =============================================
# 공통 데이터 준비 (01_필수 노트북과 동일)
# =============================================

import pandas as pd

# 설명: 이 노트북 전체에서 사용할 판매 데이터
# pandas 연결: 딕셔너리를 DataFrame으로 변환 (01_필수에서 배운 내용)
판매_딕셔너리 = {
    '상품명':   ['노트북', '마우스', '키보드', '모니터', '헤드셋', '웹캠'],
    '카테고리': ['IT기기', 'IT기기', 'IT기기', 'IT기기', '음향기기', '영상기기'],
    '가격':     [1200000, 35000, 75000, 450000, 89000, 120000],
    '수량':     [5, 50, 30, 8, 20, 12],
    '지역':     ['서울', '부산', '서울', '대구', '서울', '인천'],
    '평점':     [4.8, 4.5, 4.7, 4.6, 4.3, 4.9],
}

df = pd.DataFrame(판매_딕셔너리)
df['매출'] = df['가격'] * df['수량']  # 매출 = 가격 × 수량

print('데이터 준비 완료!')
print(df)

---
## 1장. 함수 (def)

함수는 **재사용 가능한 코드 블록**입니다.  
같은 작업을 여러 번 해야 할 때 함수로 만들면 코드가 깔끔해집니다.

> **pandas 연결**:  
> `df['컬럼'].apply(내함수)` — 컬럼의 모든 값에 함수를 적용  
> `df.apply(내함수, axis=1)` — 각 행에 함수를 적용

In [ ]:
# =============================================
# 1-1. 기본 함수 정의와 호출
# =============================================

# 설명: def 키워드로 함수 정의, return으로 결과 반환
# 구조: def 함수이름(매개변수1, 매개변수2, ...):

def 가격_등급(가격):
    """가격을 입력받아 등급 문자열을 반환하는 함수"""
    if 가격 >= 500000:
        return '프리미엄'   # 50만원 이상
    elif 가격 >= 100000:
        return '중고가'     # 10만원 이상
    elif 가격 >= 50000:
        return '중가'       # 5만원 이상
    else:
        return '저가'       # 5만원 미만

# 함수 호출 테스트
print('함수 테스트:')
print(f'  1,200,000원 → {가격_등급(1200000)}')
print(f'     35,000원 → {가격_등급(35000)}')
print(f'     89,000원 → {가격_등급(89000)}')

In [ ]:
# =============================================
# 1-2. apply()로 DataFrame 전체에 함수 적용
# =============================================

# 설명: apply()는 컬럼의 각 값에 함수를 자동으로 적용
# pandas 연결: for 루프 없이도 모든 행에 함수 적용 가능!

# 단순 apply: 컬럼 하나에 함수 적용
df['가격등급'] = df['가격'].apply(가격_등급)
print('=== 가격등급 컬럼 추가 ===')
print(df[['상품명', '가격', '가격등급']])

In [ ]:
# =============================================
# 1-3. 여러 컬럼을 동시에 사용하는 함수 + apply
# =============================================

# 설명: axis=1 옵션으로 행(row) 전체를 함수에 전달
# pandas 연결: 여러 컬럼 값을 동시에 참조하는 복잡한 계산에 사용

def 성과_평가(행):
    """각 행의 매출과 평점을 기반으로 성과를 평가"""
    매출 = 행['매출']
    평점 = 행['평점']
    
    if 매출 >= 1000000 and 평점 >= 4.5:
        return '★★★ 우수'       # 매출 높고 평점도 높음
    elif 매출 >= 500000 or 평점 >= 4.7:
        return '★★  양호'       # 하나라도 뛰어남
    else:
        return '★   보통'        # 나머지

# axis=1: 각 행을 함수에 전달 (axis=0은 각 컬럼)
df['성과'] = df.apply(성과_평가, axis=1)
print('=== 성과 평가 결과 ===')
print(df[['상품명', '매출', '평점', '성과']])

In [ ]:
# =============================================
# 1-4. 기본값 매개변수와 다양한 반환
# =============================================

# 설명: 매개변수에 기본값을 설정하면 생략 가능
# pandas 연결: pandas 함수도 대부분 기본값이 설정된 매개변수를 가짐
#              예: df.sort_values('가격', ascending=True)에서 ascending=True가 기본값

def 할인_계산(원래_가격, 할인율=0.1):
    """할인율 적용 가격을 계산. 기본 할인율은 10%"""
    할인금액 = 원래_가격 * 할인율
    할인_후_가격 = 원래_가격 - 할인금액
    return round(할인_후_가격)  # 정수로 반올림

# 기본 할인율(10%) 적용
print(f'노트북 10% 할인: {할인_계산(1200000):,}원')

# 20% 할인 적용
print(f'노트북 20% 할인: {할인_계산(1200000, 할인율=0.2):,}원')

# pandas apply + 기본값 함수
df['10%할인가'] = df['가격'].apply(할인_계산)
df['20%할인가'] = df['가격'].apply(할인_계산, 할인율=0.2)
print('\n=== 할인가 계산 결과 ===')
print(df[['상품명', '가격', '10%할인가', '20%할인가']])

---
## 2장. lambda 함수

lambda는 **이름 없는 일회용 함수**입니다.  
간단한 변환 작업을 한 줄로 표현할 때 사용합니다.

```python
# def 함수 (여러 줄)
def 두배(x):
    return x * 2

# lambda (한 줄) - 완전히 동일!
lambda x: x * 2
```

> **pandas 연결**:  
> `df['가격'].map(lambda x: x * 1.1)` — 10% 인상  
> `df['상품명'].apply(lambda x: x.upper())` — 대문자 변환

In [ ]:
# =============================================
# 2-1. lambda 기본 문법
# =============================================

# 설명: lambda 매개변수: 반환값 (return 키워드 불필요)
# 주의: lambda는 단순 표현식만 가능, 복잡한 로직은 def 사용

# 기본 lambda
두배 = lambda x: x * 2
세제곱 = lambda x: x ** 3
인사 = lambda 이름: f'안녕하세요, {이름}님!'

print('두배:', 두배(5))          # → 10
print('세제곱:', 세제곱(3))      # → 27
print('인사:', 인사('홍길동'))   # → 안녕하세요, 홍길동님!

# 매개변수 여러 개
더하기 = lambda x, y: x + y
print('더하기:', 더하기(10, 20))  # → 30

In [ ]:
# =============================================
# 2-2. lambda + map(), filter()
# =============================================

# 설명: map()은 모든 요소에 함수 적용, filter()는 조건에 맞는 요소만 추출
# pandas 연결: Series.map()과 동일한 개념!

가격_리스트 = [1200000, 35000, 75000, 450000, 89000, 120000]

# map: 모든 가격에 10% 할인 적용
할인_가격 = list(map(lambda x: int(x * 0.9), 가격_리스트))
print('10% 할인 가격:', [f'{p:,}' for p in 할인_가격])

# filter: 10만원 이상인 가격만 추출
고가_필터 = list(filter(lambda x: x >= 100000, 가격_리스트))
print('10만원 이상:', [f'{p:,}' for p in 고가_필터])

# pandas Series.map()으로 동일 작업
print('\n=== pandas map() ===')
df['할인가'] = df['가격'].map(lambda x: int(x * 0.9))
print(df[['상품명', '가격', '할인가']])

In [ ]:
# =============================================
# 2-3. lambda + 조건식 (삼항 연산자)
# =============================================

# 설명: Python의 조건식 → True면 A, False면 B 반환
# 문법: (True일 때 값) if (조건) else (False일 때 값)
# pandas 연결: 이 패턴이 lambda와 결합하여 apply()에서 자주 사용됨

# 일반 삼항 연산자
가격 = 89000
등급 = '고가' if 가격 >= 100000 else '저가'
print(f'{가격:,}원 → {등급}')  # → 저가

# lambda + 조건식
고가_여부 = lambda x: '고가' if x >= 100000 else '저가'
print(f'1,200,000원 → {고가_여부(1200000)}')
print(f'   35,000원 → {고가_여부(35000)}')

# pandas에서 활용
df['고가여부'] = df['가격'].apply(lambda x: '고가' if x >= 100000 else '저가')
print('\n=== 고가 여부 분류 ===')
print(df[['상품명', '가격', '고가여부']])

# 중첩 조건식 (가독성이 떨어지므로 복잡하면 def 사용 권장)
df['등급2'] = df['가격'].apply(
    lambda x: '프리미엄' if x >= 500000 else ('중고가' if x >= 100000 else '저가')
)
print('\n=== 3단계 등급 (lambda 중첩) ===')
print(df[['상품명', '가격', '등급2']])

---
## 3장. 슬라이싱과 인덱싱

슬라이싱은 **연속된 범위**를, 인덱싱은 **특정 위치/이름**의 데이터를 가져옵니다.

> **pandas 연결**:  
> `df.iloc[행번호, 열번호]` — **위치** 기반 선택 (integer location)  
> `df.loc[인덱스, '컬럼명']` — **이름** 기반 선택 (label based)

In [ ]:
# =============================================
# 3-1. 리스트 슬라이싱 복습 → iloc 연결
# =============================================

# 설명: 리스트 슬라이싱 [시작:끝:간격] 복습
# pandas 연결: df.iloc[]의 행/열 선택이 동일한 슬라이싱 문법 사용!

데이터 = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

print('=== 리스트 슬라이싱 ===')
print('처음 3개:', 데이터[0:3])       # → [10, 20, 30]
print('3~5번째:', 데이터[3:6])        # → [40, 50, 60]
print('끝에서 3개:', 데이터[-3:])     # → [80, 90, 100]
print('2칸씩 건너뜀:', 데이터[::2])  # → [10, 30, 50, 70, 90]

print('\n=== pandas iloc - 동일한 슬라이싱 문법 ===')
print('처음 3행:')
print(df.iloc[0:3])                   # 처음 3행 선택

print('\n처음 3행, 처음 2열:')
print(df.iloc[0:3, 0:2])              # 행 0~2, 열 0~1

In [ ]:
# =============================================
# 3-2. iloc: 위치(숫자)로 DataFrame 선택
# =============================================

# 설명: iloc = integer location, 숫자 인덱스로 위치 지정
# 주의: iloc의 끝은 포함하지 않음! iloc[0:3] → 0, 1, 2행 (3 제외)

print('=== df.iloc 활용 ===')

# 단일 값 선택
print('1번째 행, 0번째 열:', df.iloc[0, 0])     # → 노트북
print('3번째 행, 2번째 열:', df.iloc[2, 2])     # → 75000 (가격)

# 행 범위 선택
print('\n2~4번째 행:')
print(df.iloc[1:4])                              # 1, 2, 3번째 행

# 행과 열 동시 범위 선택
print('\n처음 3행, 상품명·카테고리·가격 열:')
print(df.iloc[0:3, 0:3])                         # 행 0~2, 열 0~2

# 마지막 2행
print('\n마지막 2행:')
print(df.iloc[-2:])                              # 끝에서 2행

In [ ]:
# =============================================
# 3-3. loc: 이름(레이블)으로 DataFrame 선택
# =============================================

# 설명: loc = label based, 컬럼명이나 인덱스 이름으로 선택
# 주의: loc의 끝은 포함! loc[0:3] → 0, 1, 2, 3행 (3 포함)
# 주의: iloc[0:3]은 0,1,2 / loc[0:3]은 0,1,2,3 — 다름!

print('=== df.loc 활용 ===')

# 컬럼 이름으로 선택 (가장 자주 사용)
print('상품명과 가격 컬럼만:')
print(df.loc[:, ['상품명', '가격']])  # 모든 행, 특정 컬럼만

# 행 범위 + 컬럼 이름
print('\n처음 3행의 상품명·가격·매출:')
print(df.loc[0:2, ['상품명', '가격', '매출']])  # loc에서 2는 포함!

# 조건 필터링과 loc 결합
print('\n가격 10만원 이상 상품의 상품명·가격·평점:')
print(df.loc[df['가격'] >= 100000, ['상품명', '가격', '평점']])

In [ ]:
# =============================================
# 3-4. iloc vs loc 비교 정리
# =============================================

# 설명: 둘의 차이를 명확히 이해하는 것이 중요

# 커스텀 인덱스 설정 예시
df_커스텀 = df.copy()
df_커스텀.index = ['A', 'B', 'C', 'D', 'E', 'F']  # 문자 인덱스로 변경

print('=== 커스텀 인덱스 DataFrame ===')
print(df_커스텀[['상품명', '가격']].head(3))

print('\n--- iloc[0] → 항상 첫 번째 행 ---')
print(df_커스텀.iloc[0][['상품명', '가격']])  # 위치로 첫 번째 행

print('\n--- loc["A"] → 인덱스가 A인 행 ---')
print(df_커스텀.loc['A'][['상품명', '가격']])  # 레이블 A인 행

# 결론 요약
print('\n📌 요약')
print('  iloc → 숫자 위치 (0, 1, 2...) → 인덱스가 뭐든 상관없음')
print('  loc  → 이름/레이블 기반 → 인덱스나 컬럼명으로 선택')

---
## 4장. 컴프리헨션 (Comprehension)

컴프리헨션은 **for 루프를 한 줄로** 압축하는 Python의 강력한 문법입니다.

```python
# for 루프 방식 (3줄)
결과 = []
for x in 리스트:
    결과.append(x * 2)

# 리스트 컴프리헨션 (1줄)
결과 = [x * 2 for x in 리스트]
```

> **pandas 연결**:  
> 컬럼 목록 선택, 조건부 컬럼 생성, 간단한 데이터 전처리에 활용

In [ ]:
# =============================================
# 4-1. 리스트 컴프리헨션 기본
# =============================================

# 설명: [표현식 for 변수 in 반복가능객체]
# 특징: for 루프보다 빠르고 간결 (Python 철학에 맞음)

숫자들 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# for 루프 방식
제곱_for = []
for n in 숫자들:
    제곱_for.append(n ** 2)

# 리스트 컴프리헨션 방식
제곱_comp = [n ** 2 for n in 숫자들]

print('for 루프 결과:', 제곱_for)
print('컴프리헨션 결과:', 제곱_comp)
print('결과 동일?', 제곱_for == 제곱_comp)  # → True

# 문자열 처리 예시
상품명들 = ['노트북', '마우스', '키보드', '모니터']
상품명_대문자 = [상품.upper() for 상품 in 상품명들]  # 영어 대문자 변환
상품명_길이 = [len(상품) for 상품 in 상품명들]        # 글자수
print('\n상품명 길이:', 상품명_길이)

In [ ]:
# =============================================
# 4-2. 조건부 컴프리헨션
# =============================================

# 설명: 조건을 추가하여 필터링 + 변환을 동시에
# 문법: [표현식 for 변수 in 반복가능객체 if 조건]

가격들 = [1200000, 35000, 75000, 450000, 89000, 120000]

# 10만원 이상인 가격만 추출
고가만 = [p for p in 가격들 if p >= 100000]
print('10만원 이상:', [f'{p:,}' for p in 고가만])

# 10만원 이상인 가격에 10% 할인 적용
고가_할인 = [int(p * 0.9) for p in 가격들 if p >= 100000]
print('고가 10% 할인:', [f'{p:,}' for p in 고가_할인])

# 조건부 표현식 (삼항)
등급들 = ['고가' if p >= 100000 else '저가' for p in 가격들]
print('가격 등급:', 등급들)

# pandas에서도 동일하게 활용
print('\n=== pandas에서 컴프리헨션 활용 ===')
# 특정 조건에 맞는 컬럼만 선택
숫자형_컬럼 = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
print('숫자형 컬럼들:', 숫자형_컬럼)
print(df[숫자형_컬럼].head(3))

In [ ]:
# =============================================
# 4-3. 딕셔너리 컴프리헨션
# =============================================

# 설명: 딕셔너리를 한 줄로 생성하는 문법
# 문법: {키표현식: 값표현식 for 변수 in 반복가능객체}
# pandas 연결: 컬럼별 집계 딕셔너리 생성에 활용

상품명들 = ['노트북', '마우스', '키보드', '모니터', '헤드셋', '웹캠']
가격들   = [1200000, 35000, 75000, 450000, 89000, 120000]

# zip()으로 두 리스트를 묶어 딕셔너리 생성
상품_가격_딕셔너리 = {상품: 가격 for 상품, 가격 in zip(상품명들, 가격들)}
print('상품-가격 딕셔너리:')
for k, v in 상품_가격_딕셔너리.items():
    print(f'  {k}: {v:,}원')

# 조건부 딕셔너리 컴프리헨션
고가_딕셔너리 = {상품: 가격 for 상품, 가격 in zip(상품명들, 가격들) if 가격 >= 100000}
print('\n고가 상품만:', 고가_딕셔너리)

# pandas에서 활용: 컬럼별 평균을 딕셔너리로
숫자_컬럼 = ['가격', '수량', '평점', '매출']
컬럼별_평균 = {col: df[col].mean() for col in 숫자_컬럼}
print('\n컬럼별 평균:')
for k, v in 컬럼별_평균.items():
    print(f'  {k}: {v:,.1f}')

In [ ]:
# =============================================
# 4-4. 컴프리헨션과 pandas 비교
# =============================================

# 설명: 컴프리헨션 vs pandas 벡터화 연산 비교
# 주의: 대용량 데이터에서는 pandas 벡터화 연산이 훨씬 빠름
#       컴프리헨션은 소규모 데이터나 pandas가 없는 환경에서 유용

import time
import random

# 10만 개 데이터로 성능 비교
큰_가격_리스트 = [random.randint(10000, 2000000) for _ in range(100000)]
큰_가격_시리즈 = pd.Series(큰_가격_리스트)

# 컴프리헨션 방식
시작 = time.time()
결과1 = [int(p * 0.9) for p in 큰_가격_리스트]
컴프리헨션_시간 = time.time() - 시작

# pandas 벡터화 방식
시작 = time.time()
결과2 = (큰_가격_시리즈 * 0.9).astype(int)
pandas_시간 = time.time() - 시작

print(f'컴프리헨션 소요시간: {컴프리헨션_시간:.4f}초')
print(f'pandas 소요시간:     {pandas_시간:.4f}초')
print(f'pandas가 약 {컴프리헨션_시간/max(pandas_시간, 0.0001):.1f}배 빠름')
print('\n→ 대용량 데이터는 pandas 벡터화 연산을 사용하세요!')

---
## 5장. 종합 실습

함수, lambda, 슬라이싱, 인덱싱, 컴프리헨션을 모두 활용하는 종합 분석을 수행합니다.

In [ ]:
# =============================================
# 5-1. 종합 실습: 판매 데이터 분석 대시보드
# =============================================

# 1단계: def 함수로 등급 분류
def 종합점수(행):
    """매출과 평점을 결합한 종합 점수 계산"""
    # 매출을 100만원 기준으로 정규화 (0~1 범위)
    매출_점수 = min(행['매출'] / 1000000, 1.0)
    # 평점을 5점 만점 기준으로 정규화 (0~1 범위)
    평점_점수 = 행['평점'] / 5.0
    # 매출 60%, 평점 40% 가중 평균
    return round(매출_점수 * 0.6 + 평점_점수 * 0.4, 3)

# 2단계: apply()로 종합점수 컬럼 추가
df['종합점수'] = df.apply(종합점수, axis=1)

# 3단계: lambda로 등급 분류
df['최종등급'] = df['종합점수'].apply(
    lambda x: 'S' if x >= 0.7 else ('A' if x >= 0.5 else 'B')
)

# 4단계: loc로 S등급 상품만 선택
S등급 = df.loc[df['최종등급'] == 'S', ['상품명', '매출', '평점', '종합점수', '최종등급']]
print('=== S등급 상품 ===')
print(S등급)

# 5단계: 컴프리헨션으로 요약
등급별_요약 = {등급: df[df['최종등급'] == 등급]['상품명'].tolist()
               for 등급 in ['S', 'A', 'B']}
print('\n=== 등급별 상품 목록 ===')
for 등급, 상품들 in 등급별_요약.items():
    print(f'  [{등급}등급] {상품들}')

In [ ]:
# =============================================
# 5-2. 최종 분석 결과 요약
# =============================================

# 상위 3개 상품 선택 (iloc 활용)
df_정렬 = df.sort_values('종합점수', ascending=False).reset_index(drop=True)

print('=== TOP 3 상품 (iloc로 선택) ===')
top3 = df_정렬.iloc[0:3]  # 상위 3개 행
print(top3[['상품명', '매출', '평점', '종합점수', '최종등급']])

# 숫자형 컬럼만 컴프리헨션으로 추출
숫자_컬럼들 = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
print('\n=== 숫자형 컬럼 통계 요약 ===')
print(df[숫자_컬럼들].describe().round(0))

---
## 정리

| 개념 | 핵심 문법 | pandas 연결 |
|------|-----------|-------------|
| **def 함수** | `def 함수명(매개변수): return 값` | `df.apply(함수)`, `df.apply(함수, axis=1)` |
| **lambda** | `lambda x: 표현식` | `df['열'].map(lambda x: ...)` |
| **iloc** | `df.iloc[행번호, 열번호]` | 위치 기반 선택, 숫자로 지정 |
| **loc** | `df.loc[인덱스, '컬럼명']` | 이름 기반 선택, 조건과 결합 |
| **컴프리헨션** | `[식 for x in 리스트 if 조건]` | 컬럼 선택, 조건부 변환 |

### 다음 단계
**03_중요.ipynb** → 클래스 기초, 파일 입출력, 예외 처리를 배웁니다.  
특히 클래스를 이해하면 DataFrame과 Series가 **객체(Object)**라는 점이 명확해집니다.